# 🇸🇬 Singapore AI Hub Investment Intelligence
## Multi-Agent Intelligence Framework

---

### Central Topic
**Singapore as Asia's Pioneer AI Hub** — This notebook builds a grounded multi-agent RAG + live web search system from twenty-one official documents to power an investment intelligence chatbot. The system answers critical questions for tech AI companies evaluating **where in Asia to establish their regional AI hub**.

---

### Business Context: The Investment Decision

A tech AI company is evaluating multiple Asian cities for its **regional AI hub**.
The multi-agent chatbot should address:

1. What is Singapore's National AI Strategy (NAIS 2.0) and governance framework ?
2. What practical incentives, talent pools, and infrastructure advantages does Singapore offer?
3. How does Singapore compare to neighbouring ASEAN countries on AI readiness and investment climate?
4. What cybersecurity and compliance frameworks govern LLM-based AI application deployments?
5. What are the **latest** AI news, funding rounds, and policy announcements? *(live web search)*

---

### Multi-Agent Architecture

```
┌──────────────────────────────────────────────────────────────────────────────────┐
│         INVESTOR QUERY INTERFACE  (Gradio Chatbot  /  FastAPI REST)              │
└────────────────────────────────────┬─────────────────────────────────────────────┘
                                     │
                                     ▼
          ┌──────────────────────────────────────────────────────────────────┐
          │                   ORCHESTRATOR / ROUTER                          │
          │          (LLM Intent Classifier + Keyword Fallback)              │
          │                selects 1 to 5 agents per query                   │
          └──┬────────────┬────────────┬────────────┬────────────────────┬───┘
             │            │            │            │                    │
             ▼            ▼            ▼            ▼                    ▼
      ┌───────────┐ ┌───────────┐ ┌───────────┐ ┌───────────┐  ┌──────────────┐
      │ SG Policy │ │ Invest.   │ │ Regional  │ │ Cybersec. │  │ Web Search   │
      │   Agent   │ │ Ecosystem │ │   Comp.   │ │ Compliance│  │    Agent     │
      │   (RAG)   │ │   Agent   │ │   Agent   │ │   Agent   │  │              │
      └─────┬─────┘ └─────┬─────┘ └─────┬─────┘ └─────┬─────┘  └────────┬─────┘
            │              │              │              │              │
            ▼              ▼              ▼              ▼              ▼
       ChromaDB       ChromaDB       ChromaDB       ChromaDB       DuckDuckGo
      sg_policy_     invest_eco_   regional_cmp_  cybersec_cmp_  (live internet)
      collection     collection    collection     collection
            │              │              │              │               │
            └──────────────┴──────────────┴──────────────┴───────────────┘
                                          │
                                          ▼  (all agent outputs)
                            ┌─────────────────────────────┐
                            │       SYNTHESIZER NODE      │  <- LangGraph Final Node
                            │   (Executive-Level Answer)  │
                            └─────────────┬───────────────┘
                                          │
                                          ▼
                                 Final investor answer
                           ┌──────────────┴──────────────┐
                           ▼                             ▼
                   Gradio Chatbot             FastAPI JSON Response
```

**Routing logic summary:**
- **RAG agents (4)**: Grounded in 21 source PDFs via ChromaDB — selected by topic keywords
- **`web_search_agent`** *(NEW)*: Activated for "latest", "recent", "news", "2025/2026" queries — live DuckDuckGo search
- **`general_agent`**: Fallback for off-topic or greetings — base LLM only, no retrieval
- Agents can be **combined**: e.g. *"latest EDB incentives"* → `[investment_ecosystem_agent, web_search_agent]`

---

### Data Sources (`/data_agents/`)

| Document | Knowledge Domain | Agent |
|---|---|---|
| Singapore AI Opportunity.pdf | NAIS 2.0, governance, AI policy roadmap | SG Policy Agent |
| CSET-Examining-Singapores-AI-Progress.pdf | Singapore AI capabilities assessment | SG Policy Agent |
| Model-AI-Governance-Framework-for-Generative-AI-19-June-2024.pdf | Responsible GenAI deployment guidelines | SG Policy Agent |
| EDB_Singapore-Tech-Ecosystem.pdf | Ecosystem, clusters, industries | Investment Agent |
| EDB_Guide-to-Hiring-Your-Dream-Tech-Team-in-Singapore.pdf | Talent, workforce, hiring | Investment Agent |
| Gen-AI_Artificial Intelligence and the Future of Work.pdf | GenAI impact on workforce & skills SEA | Investment Agent |
| State_of_Ai_SEA_Digital.pdf | AI state across Southeast Asia | Regional Agent |
| The AI Readiness Barometer-ASEAN AI Landscape.pdf | ASEAN AI readiness rankings | Regional Agent |
| unlocking-southeast-asias-ai-potential.pdf | SEA AI growth potential | Regional Agent |
| ASEAN-Guide-on-AI-Governance-and-Ethics.pdf | ASEAN AI governance principles | Regional Agent |
| e_conomy_sea_2025_report.pdf | Digital economy size, growth, AI trends SEA | Regional Agent |
| ERIA-One-ASEAN-Start-up-White-Paper-2024.pdf | ASEAN startup ecosystems, cross-border investment | Regional Agent |
| Accelerating_SME_AI_Adoption…Malaysia.pdf | Malaysia SME AI adoption via open source | Regional Agent |
| THE-NATIONAL-GUIDELINES-ON-AI-GOVERNANCE-ETHICS.pdf | National AI governance guidelines | Regional Agent |
| Accelerating-AI-Discussions-in-ASEAN.pdf | ASEAN AI policy discussions & priorities | Regional Agent |
| Cybersecurity_Playbook_for_LLM_Applications.pdf | LLM security, OWASP, compliance | Cybersecurity Agent |
| Companion Guide on Securing AI Systems.pdf | Securing AI infrastructure & pipelines | Cybersecurity Agent |
| Guidelines on Securing AI Systems.pdf | AI security controls & deployment | Cybersecurity Agent |
| large-language-model-starter-kit.pdf | Safe LLM implementation practices | Cybersecurity Agent |
| Securing LLM Backed Systems…Authorization.pdf | LLM authorization & access control | Cybersecurity Agent |
| Singapore Cyber Landscape 2024_2025.pdf | SG cybersecurity threat landscape | Cybersecurity Agent |
| *(live internet — any source)* | Latest news, current events, real-time data | **Web Search Agent** |


---

## Section 1 — Setup & Configuration

Install dependencies and configure API keys, embedding model, and ChromaDB.


In [1]:
# Uncomment to install dependencies in a fresh environment
# !pip install -qU langchain-groq langchain-huggingface langchain-chroma \
#               langgraph langchain-community sentence-transformers \
#               chromadb pypdf gradio python-dotenv ddgs
print('Dependencies ready.')


Dependencies ready.


In [2]:
import os
import json
from typing import TypedDict, List, Dict

import chromadb
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langgraph.graph import StateGraph, END

# ── API Key ──────────────────────────────────────────────────────────────────
load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY', '')
if not os.environ['GROQ_API_KEY']:
    raise ValueError('GROQ_API_KEY missing. Add it to your .env file.')

# ── Paths ─────────────────────────────────────────────────────────────────────
_BASE = os.path.dirname(os.path.abspath('__file__'))
DATA_AGENTS_DIR = os.path.join(_BASE, 'data_agents')
CHROMA_DB_PATH  = os.path.join(_BASE, 'chroma_db_agents')

# ── Models ────────────────────────────────────────────────────────────────────
EMBEDDING_MODEL = 'all-mpnet-base-v2'
LLM_MODEL = "llama-3.3-70b-versatile"  # PRIMARY: best quality, 131K ctx (100K TPD quota)
# Switch to one of these if daily quota is exhausted (each has its own quota):
# LLM_MODEL = "openai/gpt-oss-120b"    # BEST ALT: 120B params, production-grade
# LLM_MODEL = "openai/gpt-oss-20b"     # LIGHTER ALT: 20B params, faster
# LLM_MODEL = "llama-3.1-8b-instant"   # FASTEST: 8B params, lower quality
# (Verified active Apr 2026 — see console.groq.com/docs/models)

llm              = ChatGroq(model=LLM_MODEL, temperature=0.2)
persistent_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
embeddings       = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
splitter         = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
search_tool      = DuckDuckGoSearchRun()

print('Environment configured.')
print(f'  LLM            : {LLM_MODEL} via Groq')
print(f'  Embeddings     : {EMBEDDING_MODEL}')
print(f'  Web search     : DuckDuckGoSearchRun (live internet)')
print(f'  Data directory : {DATA_AGENTS_DIR}')
print(f'  ChromaDB path  : {CHROMA_DB_PATH}')


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Environment configured.
  LLM            : llama-3.3-70b-versatile via Groq
  Embeddings     : all-mpnet-base-v2
  Web search     : DuckDuckGoSearchRun (live internet)
  Data directory : /home/symmetry/repo/su-ntu-ctp/module5/dsai3_module5/data_agents
  ChromaDB path  : /home/symmetry/repo/su-ntu-ctp/module5/dsai3_module5/chroma_db_agents


---

## Section 2 — Knowledge Base Construction

Each PDF is loaded with `PyPDFLoader`, split into 1 000-token chunks, embedded with `all-mpnet-base-v2`, and stored in a dedicated ChromaDB collection.

| Collection | Source PDFs |
|---|---|
| `sg_policy_collection` | Singapore AI Opportunity.pdf |
| `investment_ecosystem_collection` | EDB_Singapore-Tech-Ecosystem.pdf, EDB_Guide-to-Hiring.pdf |
| `regional_comparison_collection` | State_of_Ai_SEA_Digital.pdf, AI Readiness Barometer.pdf, unlocking-sea-ai.pdf |
| `cybersecurity_compliance_collection` | Cybersecurity_Playbook_for_LLM_Applications.pdf |


In [3]:
# Map each PDF file  →  its target ChromaDB collection
PDF_COLLECTION_MAP = {
    # ── SG Policy ─────────────────────────────────────────────────────────────
    'Singapore AI Opportunity.pdf':
        'sg_policy_collection',
    'CSET-Examining-Singapores-AI-Progress.pdf':
        'sg_policy_collection',
    'Model-AI-Governance-Framework-for-Generative-AI-19-June-2024.pdf':
        'sg_policy_collection',

    # ── Investment Ecosystem ──────────────────────────────────────────────────
    'EDB_Singapore-Tech-Ecosystem.pdf':
        'investment_ecosystem_collection',
    'EDB_Guide-to-Hiring-Your-Dream-Tech-Team-in-Singapore.pdf':
        'investment_ecosystem_collection',
    'Gen-AI_Artificial Intelligence and the Future of Work _sdnea2024001.pdf':
        'investment_ecosystem_collection',

    # ── Regional Comparison ───────────────────────────────────────────────────
    'State_of_Ai_SEA_Digital.pdf':
        'regional_comparison_collection',
    'The AI Readiness Barometer-ASEAN AI Landscape.pdf':
        'regional_comparison_collection',
    'unlocking-southeast-asias-ai-potential.pdf':
        'regional_comparison_collection',
    'ASEAN-Guide-on-AI-Governance-and-Ethics_beautified_201223_v2.pdf':
        'regional_comparison_collection',
    'e_conomy_sea_2025_report.pdf':
        'regional_comparison_collection',
    'ERIA-One-ASEAN-Start-up-White-Paper-2024.pdf':
        'regional_comparison_collection',
    'Accelerating_SME_AI_Adoption_Through_Open_Source_in_Malaysia_s_Digital_Future_NAIO.pdf':
        'regional_comparison_collection',
    'THE-NATIONAL-GUIDELINES-ON-AI-GOVERNANCE-ETHICS.pdf':
        'regional_comparison_collection',
    'Accelerating-AI-Discussions-in-ASEAN-.pdf':
        'regional_comparison_collection',

    # ── Cybersecurity & Compliance ────────────────────────────────────────────
    'Cybersecurity_Playbook_for_Large_Language_Model_LLM_Applications.pdf':
        'cybersecurity_compliance_collection',
    'Companion Guide on Securing AI Systems.pdf':
        'cybersecurity_compliance_collection',
    'Guidelines on Securing AI Systems.pdf':
        'cybersecurity_compliance_collection',
    'large-language-model-starter-kit.pdf':
        'cybersecurity_compliance_collection',
    'Securing LLM Backed Systems_ Essential Authorization Practices 20240823.pdf':
        'cybersecurity_compliance_collection',
    'Singapore Cyber Landscape 2024_2025.pdf':
        'cybersecurity_compliance_collection',
}
print('PDF → Collection mapping defined.')
for pdf, col in PDF_COLLECTION_MAP.items():
    print(f'  {pdf}  -->  {col}')


PDF → Collection mapping defined.
  Singapore AI Opportunity.pdf  -->  sg_policy_collection
  CSET-Examining-Singapores-AI-Progress.pdf  -->  sg_policy_collection
  Model-AI-Governance-Framework-for-Generative-AI-19-June-2024.pdf  -->  sg_policy_collection
  EDB_Singapore-Tech-Ecosystem.pdf  -->  investment_ecosystem_collection
  EDB_Guide-to-Hiring-Your-Dream-Tech-Team-in-Singapore.pdf  -->  investment_ecosystem_collection
  Gen-AI_Artificial Intelligence and the Future of Work _sdnea2024001.pdf  -->  investment_ecosystem_collection
  State_of_Ai_SEA_Digital.pdf  -->  regional_comparison_collection
  The AI Readiness Barometer-ASEAN AI Landscape.pdf  -->  regional_comparison_collection
  unlocking-southeast-asias-ai-potential.pdf  -->  regional_comparison_collection
  ASEAN-Guide-on-AI-Governance-and-Ethics_beautified_201223_v2.pdf  -->  regional_comparison_collection
  e_conomy_sea_2025_report.pdf  -->  regional_comparison_collection
  ERIA-One-ASEAN-Start-up-White-Paper-2024.pdf  --

In [4]:
def build_knowledge_base(force_rebuild: bool = False) -> None: # True to force re-ingestion
    '''
    Ingest all PDFs in data_agents/ into domain-specific ChromaDB collections.
    Skips if all collections already exist (use force_rebuild=True to re-ingest).
    '''
    existing   = {c.name for c in persistent_client.list_collections()}
    expected   = set(PDF_COLLECTION_MAP.values())

    if not force_rebuild and expected.issubset(existing):
        print('Knowledge base already built. Skipping ingestion.')
        print('  Call build_knowledge_base(force_rebuild=True) to re-ingest.')
        return

    print('Building knowledge base from PDFs in data_agents/ ...')

    # --- Step 1: Load and chunk each PDF, grouped by target collection ---
    collection_docs: Dict[str, list] = {}
    for pdf_file, collection_name in PDF_COLLECTION_MAP.items():
        pdf_path = os.path.join(DATA_AGENTS_DIR, pdf_file)
        if not os.path.exists(pdf_path):
            print(f'  WARNING: PDF not found -> {pdf_file}')
            continue
        try:
            loader = PyPDFLoader(pdf_path)
            docs   = loader.load()
            splits = splitter.split_documents(docs)
            collection_docs.setdefault(collection_name, []).extend(splits)
            print(f'  Loaded  {pdf_file}  ->  {len(splits)} chunks  ->  {collection_name}')
        except Exception as e:
            print(f'  ERROR loading {pdf_file}: {e}')

    # --- Step 2: Build (or rebuild) each ChromaDB collection ---
    for collection_name, docs in collection_docs.items():
        try:
            # Drop existing collection when rebuilding
            try:
                persistent_client.delete_collection(collection_name)
            except Exception:
                pass

            Chroma(
                client=persistent_client,
                collection_name=collection_name,
                embedding_function=embeddings,
            ).add_documents(docs)
            print(f'  Indexed {len(docs)} chunks  ->  {collection_name!r}')
        except Exception as e:
            print(f'  ERROR indexing {collection_name}: {e}')

    print('Knowledge base build complete.')


build_knowledge_base(force_rebuild=False)


Knowledge base already built. Skipping ingestion.
  Call build_knowledge_base(force_rebuild=True) to re-ingest.


---

## Section 3 — Agent Definitions

Four specialised RAG agents are defined, each grounded in its own ChromaDB collection.
A fifth `web_search_agent` queries the live internet via DuckDuckGo for real-time information.
A sixth `general_agent` handles off-topic queries with the base LLM only.

| Agent | Persona | Knowledge Base |
|---|---|---|
| `sg_policy_agent` | Singapore AI Policy Expert | `sg_policy_collection` — 3 SG policy PDFs |
| `investment_ecosystem_agent` | Singapore Investment Advisor | `investment_ecosystem_collection` — 3 EDB/workforce PDFs |
| `regional_comparison_agent` | ASEAN AI Regional Analyst | `regional_comparison_collection` — 9 ASEAN PDFs |
| `cybersecurity_compliance_agent` | LLM Security Specialist | `cybersecurity_compliance_collection` — 6 security PDFs |
| `web_search_agent` | Live Web Research Agent | Live internet via DuckDuckGo — no ChromaDB |
| `general_agent` | General AI Assistant | LLM only — no retrieval |

**`web_search_agent` routing triggers:** queries containing "latest", "recent", "current", "news", "2025", "2026", "update", "live", "breaking", "announce", "new report", "this week/month/year", "just released"


In [5]:
# ── Agent routing table ───────────────────────────────────────────────────────
AGENT_TO_COLLECTION = {
    'sg_policy_agent':              'sg_policy_collection',
    'investment_ecosystem_agent':   'investment_ecosystem_collection',
    'regional_comparison_agent':    'regional_comparison_collection',
    'cybersecurity_compliance_agent': 'cybersecurity_compliance_collection',
}

AGENT_PERSONA = {
    'sg_policy_agent': (
        'Singapore AI Policy Expert. '
        'You specialise in Singapore National AI Strategy (NAIS 2.0), '
        'the Model AI Governance Framework (MAIGF), PDPC guidelines, '
        'MAS AI regulations, AI ethics principles, and the Smart Nation initiative.'
    ),
    'investment_ecosystem_agent': (
        'Singapore Investment and Ecosystem Advisor. '
        'You specialise in EDB investment incentives, Singapore tech clusters, '
        'available talent pools, startup grants (e.g. Enterprise Development Grant), '
        'industry transformation maps, and operational considerations for '
        'establishing a tech or AI company in Singapore.'
    ),
    'regional_comparison_agent': (
        'ASEAN AI Regional Analyst. '
        'You specialise in comparing AI readiness, policy maturity, digital '
        'infrastructure, talent availability, data centre capacity, and investment '
        'climate across Southeast Asian countries: Singapore, Malaysia, Thailand, '
        'Indonesia, Vietnam, and Philippines. You cite rankings and statistics.'
    ),
    'cybersecurity_compliance_agent': (
        'LLM Cybersecurity and Compliance Specialist. '
        'You specialise in security frameworks for AI and LLM applications, '
        'OWASP LLM Top 10 risks, prompt injection, data poisoning, model theft, '
        'PDPA compliance, and regulatory requirements for deploying generative '
        'AI systems in production environments.'
    ),
}

VALID_AGENTS = list(AGENT_TO_COLLECTION.keys()) + ['web_search_agent', 'general_agent']


# ── Keyword fallback router (used if LLM JSON parse fails) ───────────────────
def keyword_fallback_route(query: str) -> List[str]:
    q = query.lower()
    picked = []
    if any(k in q for k in ['policy', 'nais', 'governance', 'regulation', 'pdpc',
                              'mas', 'framework', 'strategy', 'law', 'legal',
                              'govern', 'ethic', 'smart nation']):
        picked.append('sg_policy_agent')
    if any(k in q for k in ['invest', 'edb', 'incentive', 'talent', 'hire', 'hiring',
                              'ecosystem', 'company', 'setup', 'establish', 'hub',
                              'cost', 'grant', 'subsid', 'workforce', 'cluster',
                              'startup', 'tech park', 'one north']):
        picked.append('investment_ecosystem_agent')
    if any(k in q for k in ['compare', 'comparison', 'malaysia', 'thailand',
                              'indonesia', 'vietnam', 'philippines', 'asean', 'sea',
                              'regional', 'other countr', 'adjacent', 'neighbour',
                              'neighbor', 'readiness', 'ranking', 'better than']):
        picked.append('regional_comparison_agent')
    if any(k in q for k in ['security', 'cybersecurity', 'llm', 'attack',
                              'prompt injection', 'compliance', 'owasp', 'risk',
                              'safe', 'vulnerab', 'poison', 'adversar']):
        picked.append('cybersecurity_compliance_agent')
    if any(k in q for k in ['latest', 'recent', 'current', 'now', 'today',
                              'news', '2025', '2026', 'update', 'live',
                              'breaking', 'announce', 'new report', 'this week',
                              'this month', 'this year', 'just released']):
        picked.append('web_search_agent')
    return picked or ['general_agent']


# ── Web search agent (live internet retrieval) ────────────────────────────────
def run_web_search_agent(query: str) -> dict:
    '''Search the internet for up-to-date information and synthesise an answer.'''
    try:
        results = search_tool.run(query)
    except Exception as e:
        return {'agent': 'web_search_agent',
                'answer': f'Web search unavailable: {e}',
                'used_docs': 0}

    if not results or not results.strip():
        return {'agent': 'web_search_agent',
                'answer': 'No web search results found for this query.',
                'used_docs': 0}

    system_prompt = (
        'You are a knowledgeable AI assistant specialising in Singapore technology, '
        'AI investment, ASEAN digital economy, and current events. '
        'Use ONLY the web search results below to answer the question accurately and concisely. '
        'Cite key facts and note the recency of information where relevant. '
        'Do not fabricate information beyond what the search results provide.\n\n'
        f'Web Search Results:\n{results}'
    )
    resp = llm.invoke([SystemMessage(content=system_prompt),
                       HumanMessage(content=query)])
    return {'agent': 'web_search_agent', 'answer': resp.content, 'used_docs': 0}


# ── RAG agent executor ────────────────────────────────────────────────────────
def run_grounded_agent(query: str, agent_name: str, k: int = 5) -> dict:
    collection_name = AGENT_TO_COLLECTION[agent_name]
    persona         = AGENT_PERSONA[agent_name]
    try:
        db   = Chroma(client=persistent_client,
                      collection_name=collection_name,
                      embedding_function=embeddings)
        docs = db.similarity_search(query, k=k)
    except Exception as e:
        return {'agent': agent_name,
                'answer': f'Knowledge base retrieval error: {e}',
                'used_docs': 0}

    context = '\n\n'.join(d.page_content for d in docs)
    if not context.strip():
        return {'agent': agent_name,
                'answer': 'No relevant information found in the knowledge base.',
                'used_docs': 0}

    system_prompt = f'''You are the {persona}

STRICT INSTRUCTIONS:
- Answer ONLY from the provided Context below.
- Cite specific data points, statistics, or policy names when present.
- If the answer is absent from Context, state:
  "This specific information is not available in my current knowledge base."
- Use bullet points or numbered lists for clarity.
- Do not fabricate information.

Context:
{context}'''

    resp = llm.invoke([SystemMessage(content=system_prompt),
                       HumanMessage(content=query)])
    return {'agent': agent_name, 'answer': resp.content, 'used_docs': len(docs)}


def run_general_agent(query: str) -> dict:
    system_prompt = (
        'You are a knowledgeable AI assistant specialising in Singapore technology '
        'and investment landscape. Answer helpfully and concisely.'
    )
    resp = llm.invoke([SystemMessage(content=system_prompt),
                       HumanMessage(content=query)])
    return {'agent': 'general_agent', 'answer': resp.content, 'used_docs': 0}


print('Agent tools and routing functions defined.')
print(f'  Valid agents: {VALID_AGENTS}')


Agent tools and routing functions defined.
  Valid agents: ['sg_policy_agent', 'investment_ecosystem_agent', 'regional_comparison_agent', 'cybersecurity_compliance_agent', 'web_search_agent', 'general_agent']


---

## Section 4 — LangGraph Multi-Agent Orchestration

The workflow consists of three LangGraph nodes:

| Node | Role |
|---|---|
| `router` | Classifies investor intent; selects one or more agents (up to 5) via structured LLM output with keyword fallback |
| `executor` | Runs each selected agent in sequence — RAG agents retrieve from ChromaDB; `web_search_agent` queries DuckDuckGo live |
| `synthesizer` | Merges all agent outputs into a single executive-level investment briefing |

The **router** first attempts LLM JSON parsing; keyword matching is the fallback.
Queries containing time-sensitive keywords ("latest", "recent", "2025", etc.) automatically include `web_search_agent`.
The **synthesizer** integrates all agent outputs — both RAG-grounded and live web — into a coherent investment brief.


In [6]:
class InvestorState(TypedDict):
    query:           str
    selected_agents: List[str]
    routing_reason:  str
    agent_outputs:   Dict[str, str]
    response:        str
    debug_log:       str

print('InvestorState TypedDict defined.')


InvestorState TypedDict defined.


In [7]:
# ── Gatekeeper node ──────────────────────────────────────────────────────────
# Pattern adapted from LangGraph5 / LangGraph6 / LangGraph7:
#   LLM classifier fires first → ALLOW or BLOCK before any RAG/search work.
#   Keeps the system focused on its stated Business Context (Investment Decision).

def gatekeeper_node(state: InvestorState) -> dict:
    '''
    Topic relevance guardrail.
    Classifies whether the query is within the Singapore/ASEAN AI investment domain.
    ALLOW → proceeds to router_node.
    BLOCK → short-circuits to blocked_response_node, no LLM/ChromaDB calls consumed.
    '''
    query = state['query']

    gatekeeper_prompt = f'''You are a topic relevance classifier for the
Singapore AI Hub Investment Intelligence system.

The system exists to help tech companies evaluate Singapore (and ASEAN) as a
regional AI hub. Answer ONLY with ALLOW or BLOCK.

ALLOWED topics:
- Singapore AI policy: NAIS 2.0, Smart Nation, MAS/PDPC regulations, AI governance
- Singapore investment: EDB incentives, grants, startup ecosystem, operational costs
- Singapore tech ecosystem: talent, hiring, data centres, One-North, tech parks
- ASEAN regional comparison: AI readiness rankings, digital economy, investment climate
  across Singapore, Malaysia, Thailand, Indonesia, Vietnam, Philippines
- Cybersecurity & compliance for AI/LLM deployments (OWASP, PDPA, GenAI frameworks)
- Latest AI news, funding rounds, policy announcements in Singapore or Southeast Asia
- General greetings or clarifying questions about what this chatbot does

BLOCK topics (clearly off-topic):
- Sports, entertainment, celebrities, music, movies, gaming
- Cooking, food, recipes, travel, lifestyle
- Medical or health advice
- Personal finance questions unrelated to Singapore AI investment
- Software debugging / coding help unrelated to AI deployment in Singapore
- Topics about regions with no connection to ASEAN AI investment

RULES:
- When uncertain → ALLOW (a broad AI or investment question may still be useful)
- Partially relevant queries → ALLOW
- Only BLOCK when the query has no plausible connection to Singapore/ASEAN AI investment

EXAMPLES:
Q: What is Singapore's NAIS 2.0?               → ALLOW
Q: How does Singapore rank on AI readiness?    → ALLOW
Q: Latest AI funding news in 2025?             → ALLOW
Q: What are OWASP LLM Top 10 risks?            → ALLOW
Q: Who won the football World Cup?             → BLOCK
Q: Give me a pasta recipe                      → BLOCK
Q: What is Python list comprehension?          → BLOCK

Output ONLY: ALLOW or BLOCK

Query: {query}'''

    result = llm.invoke([SystemMessage(content=gatekeeper_prompt)]).content.strip().upper()
    allowed = 'BLOCK' not in result   # ALLOW if response is not BLOCK

    print(f'[Gatekeeper] {"ALLOWED" if allowed else "BLOCKED"}: {query[:80]}')
    return {'allowed': allowed, 'debug_log': f'[Gatekeeper] {"ALLOWED" if allowed else "BLOCKED"}: {query[:80]}'}


def blocked_response_node(state: InvestorState) -> dict:
    '''Polite refusal for queries outside the Singapore/ASEAN AI investment domain.'''
    return {
        'response': (
            "I'm specialised in **Singapore AI hub investment intelligence** and can only help with:\n\n"
            "- 🇸🇬 Singapore AI policy (NAIS 2.0, Smart Nation, MAS/PDPC regulations)\n"
            "- EDB investment incentives, grants, and ecosystem for tech companies\n"
            "- ASEAN regional comparison (AI readiness, digital economy, investment climate)\n"
            "- Cybersecurity & compliance for AI/LLM deployments in Singapore\n"
            "- Latest AI news, funding, and policy updates in Singapore/Southeast Asia\n\n"
            "Your question appears to be outside this scope. "
            "Please ask something related to Singapore or ASEAN as an AI investment destination."
        ),
        'debug_log': state.get('debug_log', '') + '\n[Gatekeeper] Query blocked — off-topic.',
    }


print('Gatekeeper and blocked-response nodes defined.')


Gatekeeper and blocked-response nodes defined.


In [8]:
def router_node(state: InvestorState) -> dict:
    '''
    LLM-powered intent router.
    Returns selected agents and routing reason.
    Falls back to keyword matching on JSON parse failure.
    '''
    query = state['query']

    router_prompt = '''You are an investment intelligence router for a Singapore AI hub analysis system.
Select the most relevant agents for this investor query:

- sg_policy_agent:              Singapore National AI Strategy (NAIS 2.0), governance,
                                  regulatory frameworks (PDPC, MAS), AI ethics, Smart Nation
- investment_ecosystem_agent:   EDB incentives, Singapore tech talent, ecosystem,
                                  grants, operational costs, establishing a company
- regional_comparison_agent:    Comparing Singapore vs ASEAN countries (Malaysia,
                                  Thailand, Indonesia, Vietnam, Philippines) on AI readiness,
                                  infrastructure, investment climate
- cybersecurity_compliance_agent: LLM security, OWASP, prompt injection, PDPA compliance,
                                  regulatory risk for AI deployments
- web_search_agent:             Queries requiring up-to-date / real-time information:
                                  breaking news, recent announcements, live statistics,
                                  current events, or topics that may have changed after the
                                  knowledge base was built (use for "latest", "recent",
                                  "current", "news", "2025", "2026", etc.)
- general_agent:                Off-topic, greetings, or topics outside all the above domains

Return STRICT JSON only — no markdown fences:
{"agents": ["agent_name", ...], "reason": "one-sentence explanation"}

Rules:
- Include ALL relevant agents for multi-part questions.
- For comparison questions, always include regional_comparison_agent AND
  investment_ecosystem_agent together.
- For queries explicitly asking about latest/recent/current information or news,
  always include web_search_agent (alongside domain agents if also relevant).
- Avoid general_agent unless the query is clearly off-topic or a simple greeting.
- Avoid web_search_agent for stable, well-established topics already covered by the knowledge base.'''

    selected, reason = [], 'fallback keyword routing'
    try:
        raw   = llm.invoke([SystemMessage(content=router_prompt),
                            HumanMessage(content=query)]).content
        clean = raw.strip().lstrip('`').rstrip('`')
        if clean.lower().startswith('json'):
            clean = clean[4:]
        parsed   = json.loads(clean.strip())
        selected = [a for a in parsed.get('agents', []) if a in VALID_AGENTS]
        reason   = str(parsed.get('reason', '')).strip() or 'LLM routing'
        if not selected:
            selected = keyword_fallback_route(query)
    except Exception:
        selected = keyword_fallback_route(query)

    return {
        'selected_agents': selected,
        'routing_reason':  reason,
        'debug_log': f'[Router] -> {selected}  |  {reason}',
    }


In [9]:
def execute_agents_node(state: InvestorState) -> dict:
    '''Run each selected agent and collect grounded answers.'''
    query    = state['query']
    selected = state.get('selected_agents', ['general_agent'])
    logs     = [state.get('debug_log', '')]
    outputs  = {}

    for agent in selected:
        if agent == 'general_agent':
            result = run_general_agent(query)
        elif agent == 'web_search_agent':
            result = run_web_search_agent(query)
        else:
            result = run_grounded_agent(query, agent)
        outputs[agent] = result['answer']
        if agent == 'web_search_agent':
            logs.append(f'[{agent}] live internet search completed')
        else:
            logs.append(f'[{agent}] retrieved {result.get("used_docs", 0)} chunks')

    return {
        'agent_outputs': outputs,
        'debug_log': '\n'.join(logs),
    }


In [10]:
def synthesize_node(state: InvestorState) -> dict:
    '''
    Merge multi-agent outputs into a single executive-level response.
    If only one agent fired, return its answer directly.
    '''
    query   = state['query']
    outputs = state.get('agent_outputs', {})
    selected = state.get('selected_agents', [])

    if not outputs:
        return {'response': 'No response could be generated.',
                'debug_log': state.get('debug_log', '')}

    if len(outputs) == 1:
        agent = selected[0] if selected else next(iter(outputs))
        return {
            'response': outputs[agent],
            'debug_log': state.get('debug_log', '')
                         + f'\n[Synthesizer] single-agent pass-through: {agent}',
        }

    specialist_reports = '\n\n'.join(
        f'[{name}]\n{text}' for name, text in outputs.items()
    )

    synthesis_prompt = f'''You are a senior investment intelligence analyst preparing
an executive briefing for a CEO who is deciding where to establish a regional AI hub in Asia.

Your task: Synthesise the specialist reports below into ONE cohesive, executive-level answer.

Guidelines:
- Integrate insights seamlessly — do NOT simply concatenate the reports.
- Lead with Singapore\'s key competitive advantages supported by evidence.
- Use structured tables or bullet lists when comparing countries.
- Be direct, concise, and actionable — every sentence should aid the investment decision.
- Do NOT add facts beyond what the specialists reported.
- If reports conflict, acknowledge the discrepancy explicitly.

Investor Question:
{query}

Specialist Reports:
{specialist_reports}'''

    final = llm.invoke([
        SystemMessage(content=synthesis_prompt),
        HumanMessage(content='Provide the synthesised executive-level briefing.'),
    ]).content

    return {
        'response':  final,
        'debug_log': state.get('debug_log', '')
                     + f'\n[Synthesizer] merged {len(outputs)} agents: '
                     + str(list(outputs.keys())),
    }


In [11]:
workflow = StateGraph(InvestorState)

# Nodes
workflow.add_node('gatekeeper',  gatekeeper_node)
workflow.add_node('blocked',     blocked_response_node)
workflow.add_node('router',      router_node)
workflow.add_node('executor',    execute_agents_node)
workflow.add_node('synthesizer', synthesize_node)

# Entry point → gatekeeper fires first (LangGraph5/6/7 pattern)
workflow.set_entry_point('gatekeeper')

# Gatekeeper routing: ALLOW → router, BLOCK → immediate refusal
workflow.add_conditional_edges(
    'gatekeeper',
    lambda s: 'router' if s.get('allowed', True) else 'blocked',
)

# Blocked path ends immediately — no RAG/LLM calls consumed
workflow.add_edge('blocked',     END)

# Normal path
workflow.add_edge('router',      'executor')
workflow.add_edge('executor',    'synthesizer')
workflow.add_edge('synthesizer', END)

app = workflow.compile()
print('Singapore AI Hub Intelligence graph compiled.')
print('  Flow: Query -> Gatekeeper -> [blocked | Router -> Executor -> Synthesizer] -> Answer')


Singapore AI Hub Intelligence graph compiled.
  Flow: Query -> Gatekeeper -> [blocked | Router -> Executor -> Synthesizer] -> Answer


---

## Section 5 — Investor Query Testing

Sample queries covering all agent knowledge domains, including live web search.
The system will route, retrieve, and synthesise answers grounded in the PDF dataset or the live internet.

| Query Category | Routed Agent(s) | Example Question |
|---|---|---|
| Policy | `sg_policy_agent` | What is latest Singapore's National AI Strategy (NAIS 2.0) and its key pillars? |
| Investment | `investment_ecosystem_agent` | What EDB incentives are available for AI companies? |
| Comparison | `regional_comparison_agent` | Singapore vs Malaysia for regional AI hub? |
| Cybersecurity | `cybersecurity_compliance_agent` | LLM security requirements in Singapore? |
| Multi-domain | multiple agents | Full competitive analysis: Singapore vs ASEAN |
| Live / Web Search | `web_search_agent` | What are the latest AI funding announcements in Singapore 2025? |
| Live + Domain | `web_search_agent` + domain agent | What are the most recent EDB grants announced this year? |


In [12]:
INVESTOR_QUERIES = [
    # Policy
    "What is Singapore's National AI Strategy (NAIS 2.0) and what are its key pillars?",
    # Investment
    "What government incentives and EDB programs does Singapore offer to tech AI companies?",
    # Regional comparison
    "How does Singapore compare to Malaysia and Thailand in AI readiness and investment appeal?",
    # Cybersecurity
    "What are the cybersecurity requirements and compliance frameworks for deploying LLM-based AI in Singapore?",
    # Multi-domain — the flagship investor question
    "As a tech AI company looking to establish a regional AI hub in Asia, why should I choose Singapore "    "over Indonesia, Vietnam, or Malaysia? Give me a direct comparison based on key competitive advantages.",
    # Talent
    "What is the tech talent landscape in Singapore, and how competitive is hiring compared to other SEA countries?",
    # Comprehensive
    "Give me a comprehensive executive briefing on Singapore's AI competitive advantages compared to all ASEAN nations.",
    # Market opportunity
    "What are the major AI adoption trends and growth opportunities across Southeast Asia for an AI company?",
]

print(f'{len(INVESTOR_QUERIES)} investor queries ready.')


8 investor queries ready.


In [13]:
def query_hub(question: str, show_debug: bool = True) -> None:
    '''Run a single investor question through the multi-agent system.'''
    print(f'\n{"="*80}')
    print(f'QUESTION: {question}')
    print('='*80)
    result = app.invoke({'query': question})
    print(f'\n--- ANSWER ---')
    print(result['response'])
    if show_debug:
        print(f'\n--- ORCHESTRATION LOG ---')
        print(result['debug_log'])
    print()


# ── Run the flagship investor query ──────────────────────────────────────────
query_hub(INVESTOR_QUERIES[4], show_debug=True)



QUESTION: As a tech AI company looking to establish a regional AI hub in Asia, why should I choose Singapore over Indonesia, Vietnam, or Malaysia? Give me a direct comparison based on key competitive advantages.
[Gatekeeper] ALLOWED: As a tech AI company looking to establish a regional AI hub in Asia, why should 

--- ANSWER ---
**Executive Briefing: Establishing a Regional AI Hub in Asia**

As we consider establishing a regional AI hub in Asia, Singapore stands out as the premier destination due to its key competitive advantages. The country's strengths in AI innovation, digital maturity, GenAI adoption, talent, and investment make it an ideal location for our AI company.

**Key Competitive Advantages of Singapore:**

* **AI Innovation and Strategy**: Singapore is a regional pioneer in AI innovation, with a clear strategy and rising investments in technology and talent.
* **Digital Maturity**: Singapore scores 38 on the digital maturity scale, surpassing the global average and other 

In [14]:
# Uncomment to run the full suite of investor queries
# for i, q in enumerate(INVESTOR_QUERIES, start=1):
#     print(f'\n>>> Query {i}/{len(INVESTOR_QUERIES)}')
#     query_hub(q, show_debug=False)

# Or test a custom question:
# query_hub(
#     'What regulatory framework does Singapore use for data privacy in AI applications, '
#     'and how does it compare to GDPR?'
# )


---

## Section 6 — Interactive Investment Intelligence Chatbot

A Gradio chat interface with:
- **Quick-access buttons** for each suggested investor question
- **Orchestration details** expandable section showing which agents fired and why
- Clean, executive-ready response formatting


In [15]:
import gradio as gr

SUGGESTED_QUESTIONS = [
    # Original questions
    "Compare Singapore vs Malaysia and Thailand on AI readiness and investment climate.",
    "What cybersecurity and compliance frameworks govern LLM deployments in Singapore?",
    "Why choose Singapore over Indonesia or Vietnam as a regional AI hub?",
    "What is the tech talent landscape in Singapore vs other ASEAN countries?",
    "Executive briefing: Singapore's AI competitive advantages vs all ASEAN nations.",
    "What AI growth opportunities exist across Southeast Asia for a new AI company?",
    "What does the Companion Guide on Securing AI Systems recommend for protecting AI infrastructure?",
    # Questions for the 8 PDFs added in the previous session
    "How does the ASEAN Guide on AI Governance and Ethics shape responsible AI deployment across the region?",
    "How does the CSET assessment evaluate Singapore's AI progress and global competitiveness?",
    "What does the e-Conomy SEA 2025 report reveal about digital economy growth and AI opportunities across Southeast Asia?",
    "What does the ERIA One ASEAN Startup White Paper say about startup ecosystems and investment opportunities in ASEAN?",
    "How is Malaysia accelerating SME AI adoption through open source, and what does this mean for regional competitiveness?",
    "What are the key impacts of Generative AI on the future of work in Southeast Asia, and how should companies prepare?",
    "What does the Singapore Cyber Landscape 2024/2025 report reveal about key threats and Singapore's cybersecurity posture?",
    "What national AI governance and ethics guidelines are shaping AI deployment standards across the region?",
    # Questions for the 2 PDFs added in the previous session
    "What does Singapore's Model AI Governance Framework for Generative AI recommend for responsible GenAI deployment?",
    "What are the key AI policy discussions and priorities being accelerated across ASEAN member states?",
    # Live web search questions (routed to web_search_agent for up-to-date answers)
    "What are the latest AI investment announcements and funding rounds in Singapore in 2025/2026?",
    "What recent AI policy updates or new regulations has Singapore announced this year?",
]


def chat_with_intelligence(message: str) -> str:
    if not message.strip():
        return ''
    try:
        result   = app.invoke({'query': message})
        answer   = result.get('response', 'No response generated.')
        selected = result.get('selected_agents', [])
        reason   = result.get('routing_reason', '')
        log      = result.get('debug_log', '')

        debug_html = (
            '\n\n---\n<details><summary>Orchestration Details</summary>\n\n'
            f'**Agents activated:** {", ".join(selected)}\n\n'
            f'**Routing reason:** {reason}\n\n'
            f'**Debug log:**\n```\n{log}\n```\n</details>'
        )
        return f'{answer}{debug_html}'
    except Exception as e:
        return f'System error: {e}'


# ── Typography + floating sticky input bar ────────────────────────────────────
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500&display=swap');

/* ── Base: Inter across the entire UI ────────────────────────────────────── */
.gradio-container, .gradio-container * {
    font-family: 'Inter', system-ui, -apple-system, BlinkMacSystemFont,
                 'Segoe UI', sans-serif !important;
}

/* ── Heading hierarchy ────────────────────────────────────────────────────── */
.gradio-container h1 { font-size: 1.75rem; font-weight: 700; letter-spacing: -0.03em; line-height: 1.25; }
.gradio-container h2 { font-size: 1.35rem; font-weight: 600; letter-spacing: -0.02em; }
.gradio-container h3 { font-size: 1.1rem;  font-weight: 600; letter-spacing: -0.01em; }

/* ── Chat message bubbles ─────────────────────────────────────────────────── */
.message-wrap .message,
.message-wrap .message p {
    font-size: 15px !important;
    line-height: 1.75 !important;
    letter-spacing: 0.01em;
}

/* ── Code / debug log blocks ──────────────────────────────────────────────── */
.message-wrap code,
.message-wrap pre,
.message-wrap pre code {
    font-family: 'JetBrains Mono', 'Fira Code', 'Cascadia Code',
                 'Consolas', monospace !important;
    font-size: 13px !important;
    line-height: 1.65 !important;
    letter-spacing: 0.02em;
}

/* ── Buttons ──────────────────────────────────────────────────────────────── */
.gr-button        { font-size: 13px !important; font-weight: 500 !important; line-height: 1.5 !important; }
button.primary    { font-size: 15px !important; font-weight: 600 !important; }
label, .label-wrap span { font-size: 13px !important; font-weight: 500 !important; }

/* ── Floating sticky input bar ────────────────────────────────────────────── */
#input_area {
    position: fixed;
    bottom: 0;
    left: 0;
    right: 0;
    z-index: 1000;
    background: var(--background-fill-primary, #ffffff);
    padding: 12px 24px 14px;
    border-top: 1px solid var(--border-color-primary, #e5e7eb);
    box-shadow: 0 -6px 24px rgba(0, 0, 0, 0.10);
}

/* Textbox inside the sticky bar */
#input_area textarea {
    font-size: 15px !important;
    line-height: 1.6 !important;
    letter-spacing: 0.01em;
    border-radius: 8px !important;
}

/* Push the chatbot column up so fixed bar never covers messages */
#chat_col {
    padding-bottom: 130px;
}
"""

# ── Build Gradio UI ───────────────────────────────────────────────────────────
with gr.Blocks(
    theme=gr.themes.Soft(
        font=[gr.themes.GoogleFont("Inter"), "system-ui", "sans-serif"],
        font_mono=[gr.themes.GoogleFont("JetBrains Mono"), "Consolas", "monospace"],
    ),
    title='SG AI Hub Intelligence',
    css=custom_css,
) as demo:

    gr.Markdown("""
# Singapore AI Hub Investment Intelligence Chatbot
### Powered by Multi-Agent RAG + Live Web Search  |  LangGraph + ChromaDB + Groq + DuckDuckGo

Ask any question about **Singapore as an AI hub** — governance, investment incentives,
talent, regional comparisons, compliance requirements, or the **latest news and announcements**.
    """)

    with gr.Row():
        with gr.Column(scale=3, elem_id="chat_col"):
            chatbot = gr.Chatbot(
                value=[],
                height=1000,
                min_width=800,
                label='Investment Intelligence Chat',
            )
            with gr.Group(elem_id="input_area"):
                msg_box = gr.Textbox(
                    placeholder='Type your question here and press Enter or click Ask...',
                    label='Your Question',
                    lines=2,
                )
                with gr.Row():
                    submit_btn = gr.Button('Ask', variant='primary')
                    clear_btn  = gr.Button('Clear')

        with gr.Column(scale=1):
            gr.Markdown('### Suggested Questions')
            for q in SUGGESTED_QUESTIONS:
                gr.Button(q, size='sm').click(
                    fn=lambda question=q: question,
                    outputs=msg_box,
                )

    # Gradio 6.x Chatbot uses messages format: list of {'role':..., 'content':...} dicts
    def respond(message, chat_history):
        if not message.strip():
            return chat_history, ''
        bot_response = chat_with_intelligence(message)
        chat_history = chat_history + [
            {'role': 'user',      'content': message},
            {'role': 'assistant', 'content': bot_response},
        ]
        return chat_history, ''

    submit_btn.click(respond, [msg_box, chatbot], [chatbot, msg_box])
    msg_box.submit(respond, [msg_box, chatbot], [chatbot, msg_box])
    clear_btn.click(lambda: ([], ''), outputs=[chatbot, msg_box])

demo.launch(debug=True, share=False)


/tmp/ipykernel_11762/2351534921.py:118: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


[Gatekeeper] BLOCKED: How is weather today?

[Gatekeeper] ALLOWED: Who are you?
[Gatekeeper] BLOCKED: Is green house effect a cause of global warming?
[Gatekeeper] ALLOWED: Is AI data center energy consumption causing global warming to be accelerating?
Keyboard interruption in main thread... closing server.
